# Bootstrap Uncertainty Analysis

This notebook estimates uncertainty in population-based recombination times by bootstrap resampling over independent initial-condition population traces.

For each bootstrap trial, the initial-condition traces are sampled with replacement, averaged, and refitted with the same single-exponential recovery model.

In [ ]:
import csv
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

# Analysis settings
METHODS = ["FSSH", "FSSH2"]
ICONDS = list(range(0, 4000, 400))
N_BOOTSTRAP = 1000
RANDOM_SEED = 12345

FOLDER_PATTERN = "{method}_NBRA_icond_{icond}"
HDF_FILENAME = "mem_data.hdf"
TIME_DATASET = "time/data"
POPULATION_DATASET = "sh_pop_adi/data"
GROUND_STATE_INDEX = 0

AU_TO_FS = 0.02418884
FS_TO_PS = 1.0e-3
TIME_MAX_PS = 4.0

OUTPUT_PREFIX = "recombination_bootstrap"


In [ ]:
def r2_score_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot <= 0:
        return np.nan
    return 1.0 - ss_res / ss_tot


def gs_recovery_exp(time_ps, p_inf, amplitude, tau_ps):
    return p_inf - amplitude * np.exp(-time_ps / tau_ps)


def fit_gs_recovery(time_ps, population):
    p0 = float(population[0])
    tail_n = max(5, len(population) // 10)
    p_inf_guess = float(np.clip(np.mean(population[-tail_n:]), 0.0, 1.0))
    amplitude_guess = p_inf_guess - p0

    optimized, _ = curve_fit(
        gs_recovery_exp,
        time_ps,
        population,
        p0=[p_inf_guess, amplitude_guess, 10.0],
        bounds=([0.0, -1.0, 1.0e-6], [1.0, 1.0, 1.0e6]),
        maxfev=50000,
    )

    fitted_curve = gs_recovery_exp(time_ps, *optimized)
    tau_ps = float(optimized[2])
    r2 = float(r2_score_manual(population, fitted_curve))
    return tau_ps, fitted_curve, r2


def bootstrap_tau(time_ps, traces, n_bootstrap=1000, seed=12345):
    rng = np.random.default_rng(seed)
    n_icond = traces.shape[0]
    bootstrap_taus = []

    for _ in range(n_bootstrap):
        sampled_indices = rng.integers(0, n_icond, size=n_icond)
        bootstrap_average = traces[sampled_indices].mean(axis=0)

        try:
            tau_ps, _, _ = fit_gs_recovery(time_ps, bootstrap_average)
            if np.isfinite(tau_ps):
                bootstrap_taus.append(tau_ps)
        except Exception:
            pass

    bootstrap_taus = np.asarray(bootstrap_taus, dtype=float)
    if bootstrap_taus.size < 20:
        return np.nan, np.nan, bootstrap_taus

    ci_low, ci_high = np.percentile(bootstrap_taus, [2.5, 97.5])
    return float(ci_low), float(ci_high), bootstrap_taus


def load_method_traces(method):
    traces = []
    valid_iconds = []
    time_ps = None

    for icond in ICONDS:
        folder = Path(FOLDER_PATTERN.format(method=method, icond=icond))
        hdf_path = folder / HDF_FILENAME

        try:
            with h5py.File(hdf_path, "r") as handle:
                current_time_ps = (
                    np.asarray(handle[TIME_DATASET], dtype=float)
                    * AU_TO_FS
                    * FS_TO_PS
                )
                populations = np.asarray(handle[POPULATION_DATASET], dtype=float)
        except Exception as error:
            print(f"Skipping {hdf_path}: {error}")
            continue

        if populations.ndim != 2 or GROUND_STATE_INDEX >= populations.shape[1]:
            print(f"Skipping {hdf_path}: unexpected population shape {populations.shape}")
            continue

        trace = populations[:, GROUND_STATE_INDEX]

        if time_ps is None:
            time_ps = current_time_ps
        elif len(current_time_ps) != len(time_ps) or not np.allclose(current_time_ps, time_ps):
            print(f"Skipping {hdf_path}: inconsistent time grid")
            continue

        traces.append(trace)
        valid_iconds.append(icond)

    if time_ps is None or not traces:
        raise RuntimeError(f"No valid traces found for {method}")

    return time_ps, np.asarray(traces), valid_iconds


In [ ]:
figure, axes = plt.subplots(
    len(METHODS),
    1,
    figsize=(8.6, 3.8 * len(METHODS)),
    sharex=True,
)
axes = np.atleast_1d(axes).flatten()

summary_rows = []

for index, method in enumerate(METHODS):
    axis = axes[index]
    time_ps, traces, valid_iconds = load_method_traces(method)
    average_trace = traces.mean(axis=0)

    for trace in traces:
        axis.plot(time_ps, trace, linewidth=0.7, alpha=0.35)

    axis.plot(time_ps, average_trace, linewidth=2.2, label="Average")

    tau_ps, fitted_curve, r2 = fit_gs_recovery(time_ps, average_trace)
    ci_low, ci_high, bootstrap_taus = bootstrap_tau(
        time_ps,
        traces,
        n_bootstrap=N_BOOTSTRAP,
        seed=RANDOM_SEED,
    )

    axis.plot(time_ps, fitted_curve, linewidth=2.2, label="Fit")
    axis.text(
        0.03,
        0.93,
        (
            f"$\\tau_{{\\mathrm{{rec}}}}$ = {tau_ps:.2f} ps\n"
            f"95% CI: {ci_low:.2f}–{ci_high:.2f} ps\n"
            f"$R^2$ = {r2:.3f}"
        ),
        transform=axis.transAxes,
        ha="left",
        va="top",
        bbox={
            "boxstyle": "round,pad=0.25",
            "facecolor": "white",
            "edgecolor": "0.8",
        },
    )

    axis.set_title(method)
    axis.set_ylabel("Ground-State Population")
    axis.set_xlim(0.0, TIME_MAX_PS)

    summary_rows.append({
        "method": method,
        "status": "ok",
        "n_icond": len(valid_iconds),
        "tau_rec_ps": f"{tau_ps:.8f}",
        "ci_low_ps": f"{ci_low:.8f}",
        "ci_high_ps": f"{ci_high:.8f}",
        "r2": f"{r2:.8f}",
    })

axes[-1].set_xlabel("Time (ps)")
figure.tight_layout()

pdf_path = Path(f"{OUTPUT_PREFIX}.pdf")
png_path = Path(f"{OUTPUT_PREFIX}.png")
csv_path = Path(f"{OUTPUT_PREFIX}_summary.csv")

figure.savefig(pdf_path, bbox_inches="tight")
figure.savefig(png_path, dpi=600, bbox_inches="tight")

with csv_path.open("w", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "method",
            "status",
            "n_icond",
            "tau_rec_ps",
            "ci_low_ps",
            "ci_high_ps",
            "r2",
        ],
    )
    writer.writeheader()
    writer.writerows(summary_rows)

plt.show()

print(f"Saved: {pdf_path}")
print(f"Saved: {png_path}")
print(f"Saved: {csv_path}")
